In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer  # Importa PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score, hinge_loss
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA




import import_ipynb
import nbimporter
from utils import prepareTrainingSet30Sec, standardization



NameError: name 'prepareTrainingSet30Sec' is not defined

In [ ]:


for scaler in scalers:
    
    knn = KNeighborsClassifier(n_neighbors=11)

    # Addestriamo il modello
    knn.fit(X_train, y_train)

    # Prediciamo sul set di test
    y_pred = knn.predict(X_test)

    # Calcoliamo l'accuratezza e il report di classificazione
    accuracy = accuracy_score(y_test, y_pred)

    # Stampiamo i risultati
    print(f"Accuratezza: {accuracy * 100:.2f}%")


Accuratezza: 29.68%
Accuratezza: 29.68%
Accuratezza: 29.68%
Accuratezza: 29.68%


In [16]:
# Definiamo una griglia di valori per n_neighbors
param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15, 20]}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Miglior numero di vicini
print("Miglior numero di vicini:", grid_search.best_params_['n_neighbors'])

Miglior numero di vicini: 20


In [26]:
# 1. Standardizziamo le feature
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 2. Applichiamo la PCA per ridurre le dimensioni (ad esempio a 10 componenti principali)
pca = PCA(n_components=15)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# 3. Ottimizziamo il numero di vicini usando GridSearchCV
param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15, 20]}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid_search.fit(X_train_pca, y_train)

# Miglior numero di vicini
best_k = grid_search.best_params_['n_neighbors']
print("Miglior numero di vicini:", best_k)

# 4. Addestriamo il modello KNN con il miglior numero di vicini trovato
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_pca, y_train)

# 5. Prediciamo sul set di test
y_pred = knn.predict(X_test_pca)

# Calcoliamo l'accuratezza e il report di classificazione
accuracy = accuracy_score(y_test, y_pred)

# Stampiamo i risultati
print(f"Accuratezza: {accuracy * 100:.2f}%")


Miglior numero di vicini: 3
Accuratezza: 82.53%


In [27]:
# Aggiungi pesi ai vicini e usa una diversa metrica
knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='manhattan')

# Ottimizza con più fold nella cross-validation
grid_search = GridSearchCV(knn, param_grid, cv=10)
grid_search.fit(X_train_pca, y_train)

# Addestriamo il modello KNN con il miglior numero di vicini trovato
best_knn = grid_search.best_estimator_

# Prediciamo sul set di test
y_pred = best_knn.predict(X_test_pca)

# Calcoliamo l'accuratezza e il report di classificazione
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuratezza: {accuracy * 100:.2f}%")


Accuratezza: 83.33%


In [31]:
# Standardizzazione con MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



# Ottimizzazione di n_neighbors
param_grid = {'n_neighbors': list(range(1, 50, 2))}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=10)
grid_search.fit(X_train_pca, y_train)

# Miglior numero di vicini
best_k = grid_search.best_params_['n_neighbors']
print("Miglior numero di vicini:", best_k)

# KNN con pesi e metrica minkowski
knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='minkowski')
knn.fit(X_train_pca, y_train)

# Predizione e calcolo dell'accuratezza
y_pred = knn.predict(X_test_pca)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuratezza: {accuracy * 100:.2f}%")

Miglior numero di vicini: 1
Accuratezza: 93.24%


In [20]:
# Caricare i dati dal CSV
file_path = 'Data/features_30_sec.csv'
data = pd.read_csv(file_path)

# Stampa i nomi delle colonne per verificare
#print("Colonne nel DataFrame:", data.columns)

# Separare le caratteristiche e le etichette
X = data.drop(columns=['filename', 'label'])  # Rimuovi filename e label
y = data['label']  # Etichette

# Suddividere il dataset in set di addestramento e test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 1. Standardizziamo le feature
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 2. Applichiamo la PCA per ridurre le dimensioni (ad esempio a 10 componenti principali)
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# 3. Ottimizziamo il numero di vicini usando GridSearchCV
param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15, 20]}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid_search.fit(X_train_pca, y_train)

# Miglior numero di vicini
best_k = grid_search.best_params_['n_neighbors']
print("Miglior numero di vicini:", best_k)

# 4. Addestriamo il modello KNN con il miglior numero di vicini trovato
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_pca, y_train)

# 5. Prediciamo sul set di test
y_pred = knn.predict(X_test_pca)

# Calcoliamo l'accuratezza e il report di classificazione
accuracy = accuracy_score(y_test, y_pred)

# Stampiamo i risultati
print(f"Accuratezza: {accuracy * 100:.2f}%")

Miglior numero di vicini: 3
Accuratezza: 57.50%
